In [0]:
# CELL 1: SETUP & WIDGETS
# -------------------------------------------------------------------------
# This cell creates the dropdowns at the top of the notebook - we will use these 
# for specific plant and material selections in specific visualisation steps 
# where we want to focus in on specific data areas. 
# Can be expanded as we build more
# Run this once. If you don't see the widgets, run it again.
# -------------------------------------------------------------------------

dbutils.widgets.text("target_catalog", "sample_synthetic_sap", "Catalog")
dbutils.widgets.text("target_schema", "sap", "Schema")

# Default values for initial run
dbutils.widgets.text("selected_plant", "1000", "1. Select Plant")
dbutils.widgets.text("selected_material", "", "2. Select Material")

CATALOG = dbutils.widgets.get("target_catalog")
SCHEMA = dbutils.widgets.get("target_schema")

print(f"Connected to: {CATALOG}.{SCHEMA}")
print("Widgets initialized. Please check the top of the notebook.")

In [0]:
# CELL 2: LIBRARIES & DATA PREP
# -------------------------------------------------------------------------
# Import necessary visualization and data processing libraries.
# -------------------------------------------------------------------------
import plotly.graph_objects as go
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [0]:
# CELL: VIEW BOM STRUCTURE (RECIPES)
# -------------------------------------------------------------------------
# Goal: Join the three key BOM tables to show the full recipe for each product.
# Logic: MAST (Material->BOM) -> STKO (Header) -> STPO (Components)
# Widets: Uses the Catalog and Schema widgets above
# -------------------------------------------------------------------------

# 1. Load Tables
df_mast = spark.table(f"{CATALOG}.{SCHEMA}.mast")  # Link: Material -> BOM ID
df_stko = spark.table(f"{CATALOG}.{SCHEMA}.stko")  # Header: Base Qty, Status
df_stpo = spark.table(f"{CATALOG}.{SCHEMA}.stpo")  # Items: Components, Qty
df_makt = spark.table(f"{CATALOG}.{SCHEMA}.makt")  # Descriptions

# 2. Execute Join
df_bom_flat = (
    df_mast.alias("link")
    # Join to Header to get Validity and Base Quantity
    .join(df_stko.alias("head"), 
          on=F.col("link.STLNR") == F.col("head.STLNR"), 
          how="inner")
    # Join to Items to get the Component List
    .join(df_stpo.alias("item"), 
          on=F.col("link.STLNR") == F.col("item.STLNR"), 
          how="inner")
    # Optional: Get English Description for the Parent
    .join(df_makt.alias("desc"), 
          (F.col("link.MATNR") == F.col("desc.MATNR")) & 
          (F.col("desc.SPRAS") == 'EN'), 
          how="left")
    .select(
        F.col("link.WERKS").alias("Plant"),
        F.col("link.MATNR").alias("Parent_Material"),
        F.col("desc.MAKTX").alias("Parent_Description"),
        F.col("link.STLNR").alias("BOM_ID"),
        F.col("head.BMENG").alias("Base_Qty"),
        F.col("head.BMEIN").alias("Base_UoM"),
        F.col("item.POSNR").alias("Item_Number"),
        F.col("item.IDNRK").alias("Component_Material"),
        F.col("item.MENGE").alias("Component_Qty"),
        F.col("item.MEINS").alias("Component_UoM")
    )
    .orderBy("Parent_Material", "Item_Number")
)

display(df_bom_flat)

In [0]:
# CELL: REVIEW DEMAND PLANS (PBED)
# -------------------------------------------------------------------------
# Goal: Review Planned Independent Requirements (PIRs) for Finished Goods.
# Logic: PBED_SIM (Demand) -> MAKT (Descriptions)
# -------------------------------------------------------------------------

# 1. Load Tables
df_pbed = spark.table(f"{CATALOG}.{SCHEMA}.pbed_sim")
df_makt = spark.table(f"{CATALOG}.{SCHEMA}.makt")

# 2. Join and Enhance Data
# We filter for Version '00' (Active Plan) usually, but here we show all.
df_demand_view = (
    df_pbed.alias("pbed")
    .join(df_makt.alias("desc"), 
          (F.col("pbed.MATNR") == F.col("desc.MATNR")) & 
          (F.col("desc.SPRAS") == 'EN'), 
          how="left")
    .withColumn("Requirement_Date", F.to_date(F.col("pbed.BDDAT"), "yyyyMMdd"))
    .withColumn("Month_Start", F.date_trunc("month", F.col("Requirement_Date")))
    .select(
        F.col("pbed.WERKS").alias("Plant"),
        F.col("pbed.MATNR").alias("Material"),
        F.col("desc.MAKTX").alias("Description"),
        F.col("pbed.VERSB").alias("Version"), # 00 = Active, 01 = Historical
        F.col("Requirement_Date"),
        F.col("Month_Start"),
        F.col("pbed.PLNMG").alias("Planned_Qty")
    )
)

print("--- Detailed Demand Plan List ---")
display(df_demand_view.orderBy("Plant", "Material", "Requirement_Date"))

# 3. Create Monthly Pivot (Aggregate View)
# Useful for high-level planning review
df_monthly_pivot = (
    df_demand_view
    .groupBy("Plant", "Material", "Description", "Version")
    .pivot("Month_Start")
    .sum("Planned_Qty")
    .orderBy("Plant", "Material", "Version")
)

print("\n--- Monthly Demand Bucket View ---")
display(df_monthly_pivot)

In [0]:
# CELL: REVIEW PRODUCTION PLANS (PLAF)
# -------------------------------------------------------------------------
# Goal: Review Planned Orders generated by the simulation.
# Logic: PLAF (Planned Orders) -> MAKT (Descriptions)
# -------------------------------------------------------------------------

# 1. Load Tables
df_plaf = spark.table(f"{CATALOG}.{SCHEMA}.plaf")
df_makt = spark.table(f"{CATALOG}.{SCHEMA}.makt")

# 2. Join and Enhance Data
# We convert strings to actual dates for sorting/grouping
df_production_view = (
    df_plaf.alias("plan")
    .join(df_makt.alias("desc"), 
          (F.col("plan.MATNR") == F.col("desc.MATNR")) & 
          (F.col("desc.SPRAS") == 'EN'), 
          how="left")
    .withColumn("Start_Date", F.to_date(F.col("plan.PSTTR"), "yyyyMMdd"))
    .withColumn("Finish_Date", F.to_date(F.col("plan.PEDTR"), "yyyyMMdd"))
    .withColumn("Week_Start", F.date_trunc("week", F.col("Start_Date")))
    .select(
        F.col("plan.PLNUM").alias("Planned_Order_ID"),
        F.col("plan.WERKS").alias("Plant"),
        F.col("plan.MATNR").alias("Material"),
        F.col("desc.MAKTX").alias("Description"),
        F.col("Start_Date"),
        F.col("Finish_Date"),
        F.col("Week_Start"),
        F.col("plan.GSMNG").alias("Target_Qty"),
        F.col("plan.BESKZ").alias("Procurement_Type") # E = In-house, F = External
    )
)

print("--- Detailed Production Schedule (Next 500 Orders) ---")
display(df_production_view.orderBy("Start_Date", "Plant", "Material").limit(500))

# 3. Create Weekly Production Load View (Aggregate)
# This shows the production schedule (Sawtooth effect)
df_weekly_production = (
    df_production_view
    .groupBy("Plant", "Week_Start", "Material", "Description")
    .agg(F.sum("Target_Qty").alias("Total_Production_Qty"))
    .orderBy("Plant", "Week_Start", "Material")
)

print("\n--- Weekly Production Load (Batching View) ---")
display(df_weekly_production)

In [0]:
# CELL: REVIEW SAFETY STOCK TARGETS (MARC)
# -------------------------------------------------------------------------
# Goal: Review Safety Stock (EISBE) levels per Material/Plant.
# Logic: MARC (Plant Data) -> MAKT (Descriptions)
# -------------------------------------------------------------------------

import pyspark.sql.functions as F

# 1. Load Tables
df_marc = spark.table(f"{CATALOG}.{SCHEMA}.marc")
df_makt = spark.table(f"{CATALOG}.{SCHEMA}.makt")

# 2. Join and Enhance Data
# EISBE is the standard SAP field for Safety Stock
df_safety_stock_view = (
    df_marc.alias("plant")
    .join(df_makt.alias("desc"), 
          (F.col("plant.MATNR") == F.col("desc.MATNR")) & 
          (F.col("desc.SPRAS") == 'EN'), 
          how="left")
    .select(
        F.col("plant.WERKS").alias("Plant"),
        F.col("plant.MATNR").alias("Material"),
        F.col("desc.MAKTX").alias("Description"),
        F.col("plant.EISBE").alias("Safety_Stock_Target"),
        F.col("plant.DISPO").alias("MRP_Controller"),
        F.col("plant.BESKZ").alias("Procurement_Type") # E = Make, F = Buy
    )
    .filter(F.col("Safety_Stock_Target") > 0) # Only show relevant items
    .orderBy(F.col("Safety_Stock_Target").desc())
)

print("--- Safety Stock Targets (Top 50 High Value Items) ---")
display(df_safety_stock_view.limit(50))

# 3. Summary by Plant
# Shows total stock buffer holding per location
df_plant_summary = (
    df_safety_stock_view
    .groupBy("Plant")
    .agg(
        F.count("Material").alias("Count_Materials"),
        F.sum("Safety_Stock_Target").alias("Total_Safety_Stock_Units")
    )
    .orderBy("Plant")
)

print("\n--- Safety Stock Summary by Plant ---")
display(df_plant_summary)

In [0]:
# ANALYSIS CELL 
# Goal: Identify largest Product/Location combinations (Finished Packs only).
# Shows Incoming vs Outgoing volumes over the past 12 months.
# -------------------------------------------------------------------------

# 1. Load Data
df_mara = spark.table(f"{CATALOG}.{SCHEMA}.mara")
df_matdoc = spark.table(f"{CATALOG}.{SCHEMA}.matdoc")

# 2. Filter for Finished Goods (FERT) only
# We only want to track "Packs" (Finished Goods)
fert_materials = df_mara.filter(F.col("MTART") == "FERT").select("MATNR")

# 3. Aggregate Volumes from Material Documents
# Logic: 
#   - SHKZG = 'S' (Debit) -> Incoming (Production/Receipts)
#   - SHKZG = 'H' (Credit) -> Outgoing (Sales/Issues)
df_matrix = (
    df_matdoc
    .join(fert_materials, "MATNR", "inner")
    .filter(F.col("BUDAT") >= F.date_format(F.add_months(F.current_date(), -12), "yyyyMMdd")) # Last 12 Months
    .groupBy("MATNR", "WERKS")
    .agg(
        F.sum(F.when(F.col("SHKZG") == "S", F.col("MENGE")).otherwise(0)).alias("Total_Incoming_Vol"),
        F.sum(F.when(F.col("SHKZG") == "H", F.col("MENGE")).otherwise(0)).alias("Total_Outgoing_Vol")
    )
    .withColumn("Total_Activity", F.col("Total_Incoming_Vol") + F.col("Total_Outgoing_Vol"))
    .orderBy(F.col("Total_Activity").desc())
)

# 4. Display for Selection
print("Top Product/Location Combinations by Volume (Last 12 Months):")
display(df_matrix)

In [0]:
# CELL 4: WEEKLY INVENTORY TREND vs. SAFETY STOCK (BWART LOGIC)
# -------------------------------------------------------------------------
# Goal: Plot WEEKLY inventory levels + Weekly Movement Bars
# UPDATED: Includes logic to calculate Opening Balance if the chart window
#          starts after the initial stock upload.
# -------------------------------------------------------------------------

import pyspark.sql.functions as F
from pyspark.sql.window import Window
import plotly.graph_objects as go
from datetime import datetime

# 1. Get Selections
sel_plant = dbutils.widgets.get("selected_plant")
sel_material = dbutils.widgets.get("selected_material")

# --- CONFIGURATION: ANALYSIS START DATE ---
# If you want to see all history, leave this None. 
# If you filter the chart (e.g. "2025-01-01"), we need to calc the stock before this date.
ANALYSIS_START_DATE = None 
# ------------------------------------------

if not sel_material or not sel_plant:
    print("Please enter a Material and Plant in the widgets above and run this cell again.")
else:
    # 2. Get Safety Stock Level (MARC)
    df_marc = spark.table(f"{CATALOG}.{SCHEMA}.marc")
    try:
        safety_stock_val = df_marc.filter(
            (F.col("MATNR") == sel_material) & (F.col("WERKS") == sel_plant)
        ).select("EISBE").collect()[0]['EISBE']
    except IndexError:
        safety_stock_val = 0
        print(f"Warning: No Master Data found for {sel_material} at {sel_plant}. Safety Stock set to 0.")

    # 3. Prepare Movements Data
    # 101/561 = In, 601/641/261 = Out
    inbound_types = ['101', '561', '501', '102'] 
    outbound_types = ['601', '641', '261', '201']

    # Base Dataframe of ALL history
    df_all_moves = (
        spark.table(f"{CATALOG}.{SCHEMA}.matdoc")
        .filter((F.col("MATNR") == sel_material) & (F.col("WERKS") == sel_plant))
        .withColumn("Date_Obj", F.to_date(F.col("BUDAT"), "yyyyMMdd"))
        .withColumn("Flow_Qty", 
                    F.when(F.col("BWART").isin(inbound_types), F.col("MENGE")) 
                     .when(F.col("BWART").isin(outbound_types), F.col("MENGE") * -1) 
                     .otherwise(0) 
        )
    )

    # 4. Handle Opening Balance Logic
    # If we filter by date, we sum up everything BEFORE that date to get "Opening Stock"
    if ANALYSIS_START_DATE:
        start_date_dt = datetime.strptime(ANALYSIS_START_DATE, "%Y-%m-%d").date()
        
        opening_balance_row = df_all_moves.filter(F.col("Date_Obj") < start_date_dt).agg(F.sum("Flow_Qty").alias("Opening_Bal")).collect()
        opening_balance = opening_balance_row[0]["Opening_Bal"] if opening_balance_row[0]["Opening_Bal"] else 0.0
        
        # Filter moves for the chart to only be AFTER start date
        df_chart_moves = df_all_moves.filter(F.col("Date_Obj") >= start_date_dt)
    else:
        # No date filter? Opening balance is effectively 0 relative to the start of time
        opening_balance = 0.0
        df_chart_moves = df_all_moves

    # 5. Aggregate Weekly
    df_weekly = (
        df_chart_moves
        .withColumn("Week_Start", F.date_trunc("week", F.col("Date_Obj")))
        .groupBy("Week_Start")
        .agg(
            F.sum(F.when(F.col("Flow_Qty") > 0, F.col("Flow_Qty")).otherwise(0)).alias("Weekly_Incoming"),
            F.sum(F.when(F.col("Flow_Qty") < 0, F.col("Flow_Qty")).otherwise(0)).alias("Weekly_Outgoing"),
            F.sum("Flow_Qty").alias("Weekly_Net")
        )
        .orderBy("Week_Start")
    )

    # 6. Calculate Running Total (Cumulative Sum)
    # We add the 'opening_balance' to the cumulative sum of the window
    window_spec = Window.orderBy("Week_Start").rowsBetween(Window.unboundedPreceding, Window.currentRow)
    
    df_inventory_trend = df_weekly.withColumn(
        "Inventory_Level", 
        F.lit(opening_balance) + F.sum("Weekly_Net").over(window_spec)
    )

    # Convert to Pandas for Plotting
    pdf = df_inventory_trend.toPandas()

    # 7. Create Chart
    if pdf.empty:
        # Special case: If no moves in period, but we have opening balance, show a straight line
        if opening_balance > 0:
            print(f"No movements in period, but Opening Balance is {opening_balance}")
            # Logic to handle empty plot if needed, or just warn
        else:
            print(f"No transactions found for Material {sel_material} in Plant {sel_plant}.")
    else:
        fig = go.Figure()

        # Demand (Red)
        fig.add_trace(go.Bar(
            x=pdf['Week_Start'], y=pdf['Weekly_Outgoing'],
            name='Demand (Issues)', marker_color='salmon', opacity=0.7
        ))

        # Supply (Green)
        fig.add_trace(go.Bar(
            x=pdf['Week_Start'], y=pdf['Weekly_Incoming'],
            name='Supply (Receipts)', marker_color='mediumseagreen', opacity=0.7
        ))

        # Stock Line (Blue)
        fig.add_trace(go.Scatter(
            x=pdf['Week_Start'], y=pdf['Inventory_Level'],
            mode='lines+markers', name='Stock Balance',
            line=dict(color='royalblue', width=3), marker=dict(size=6)
        ))

        # Safety Stock Line (Red Dash)
        fig.add_trace(go.Scatter(
            x=pdf['Week_Start'], y=[safety_stock_val] * len(pdf),
            mode='lines', name=f'Safety Stock ({int(safety_stock_val)})',
            line=dict(color='firebrick', width=2, dash='dash')
        ))

        # Update Layout with Opening Balance Annotation
        fig.update_layout(
            title=f"Inventory Dynamics: {sel_material} @ {sel_plant} (Opening Bal: {int(opening_balance)})",
            xaxis_title="Week Commencing",
            yaxis_title="Quantity",
            template="plotly_white",
            hovermode="x unified",
            barmode='overlay'
        )

        fig.show()

In [ ]:
# CELL: SUPPLY CHAIN VALUATION (GBP) - Current & Trends
# -------------------------------------------------------------------------
# Goal: Calculate the total value of inventory across the supply chain in GBP
#       1. Current value (as of today)
#       2. Weekly trend by Material Type (FERT, HALB, ROH)
#       3. Weekly trend by Location (Plant)
# Logic: MATDOC (movements) + MBEW (standard costs) = Inventory Value
# -------------------------------------------------------------------------

import pyspark.sql.functions as F
from pyspark.sql.window import Window
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Load Tables
df_matdoc = spark.table(f"{CATALOG}.{SCHEMA}.matdoc")
df_mbew = spark.table(f"{CATALOG}.{SCHEMA}.mbew")  # Material Valuation (costs)
df_mara = spark.table(f"{CATALOG}.{SCHEMA}.mara")  # Material Master (type)
df_makt = spark.table(f"{CATALOG}.{SCHEMA}.makt")  # Descriptions

# =============================================================================
# PART 1: CURRENT STOCK VALUE (AS OF TODAY)
# =============================================================================

# Calculate Current Stock Position by Material/Plant
df_stock_position = (
    df_matdoc
    .withColumn("Qty_Signed", 
                F.when(F.col("SHKZG") == "S", F.col("MENGE"))
                 .otherwise(F.col("MENGE") * -1))
    .groupBy("MATNR", "WERKS")
    .agg(F.sum("Qty_Signed").alias("Current_Stock_Qty"))
    .filter(F.col("Current_Stock_Qty") > 0)
)

# Join with MBEW and MARA
df_valued_stock = (
    df_stock_position.alias("stock")
    .join(df_mbew.alias("val"), 
          (F.col("stock.MATNR") == F.col("val.MATNR")) & 
          (F.col("stock.WERKS") == F.col("val.BWKEY")), 
          how="left")
    .join(df_mara.alias("mat"), F.col("stock.MATNR") == F.col("mat.MATNR"), how="left")
    .select(
        F.col("stock.WERKS").alias("Plant"),
        F.col("stock.MATNR").alias("Material"),
        F.col("mat.MTART").alias("Material_Type"),
        F.col("stock.Current_Stock_Qty").alias("Stock_Qty"),
        F.coalesce(F.col("val.STPRS"), F.lit(0)).alias("Standard_Price_GBP"),
        (F.col("stock.Current_Stock_Qty") * F.coalesce(F.col("val.STPRS"), F.lit(0))).alias("Stock_Value_GBP")
    )
)

print("=" * 70)
print("SUPPLY CHAIN VALUATION - CURRENT STATE")
print("=" * 70)

print("\n--- Current Value by Plant (GBP) ---")
df_plant_value = (
    df_valued_stock
    .groupBy("Plant")
    .agg(F.count("Material").alias("SKU_Count"), F.sum("Stock_Value_GBP").alias("Total_Value_GBP"))
    .orderBy("Plant")
)
display(df_plant_value)

print("\n--- Current Value by Material Type (GBP) ---")
df_type_value = (
    df_valued_stock
    .groupBy("Material_Type")
    .agg(F.count("Material").alias("SKU_Count"), F.sum("Stock_Value_GBP").alias("Total_Value_GBP"))
    .orderBy(F.col("Total_Value_GBP").desc())
)
display(df_type_value)

total_value = df_valued_stock.agg(F.sum("Stock_Value_GBP")).collect()[0][0] or 0
print(f"\n{'='*70}")
print(f"TOTAL CURRENT SUPPLY CHAIN VALUE: £{total_value:,.2f} GBP")
print(f"{'='*70}")

# =============================================================================
# PART 2: WEEKLY TRENDS - SETUP
# =============================================================================

# Get price and material type lookup
df_prices = (
    df_mbew.alias("v")
    .join(df_mara.alias("m"), F.col("v.MATNR") == F.col("m.MATNR"), how="left")
    .select(
        F.col("v.MATNR"),
        F.col("v.BWKEY").alias("WERKS"),
        F.coalesce(F.col("v.STPRS"), F.lit(0)).alias("Price"),
        F.col("m.MTART").alias("Material_Type")
    )
)

# Calculate weekly stock movements
df_weekly_moves = (
    df_matdoc
    .withColumn("Week_End", F.date_trunc("week", F.to_date(F.col("BUDAT"), "yyyyMMdd")) + F.expr("INTERVAL 6 DAYS"))
    .withColumn("Qty_Signed", 
                F.when(F.col("SHKZG") == "S", F.col("MENGE"))
                 .otherwise(F.col("MENGE") * -1))
    .groupBy("MATNR", "WERKS", "Week_End")
    .agg(F.sum("Qty_Signed").alias("Weekly_Movement"))
)

# Calculate running stock balance
window_spec = Window.partitionBy("MATNR", "WERKS").orderBy("Week_End").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_weekly_stock = (
    df_weekly_moves
    .withColumn("Stock_Balance", F.sum("Weekly_Movement").over(window_spec))
    .filter(F.col("Stock_Balance") > 0)
    .join(df_prices, on=["MATNR", "WERKS"], how="left")
    .withColumn("Stock_Value", F.col("Stock_Balance") * F.coalesce(F.col("Price"), F.lit(0)))
)

# =============================================================================
# PART 2A: WEEKLY TREND BY MATERIAL TYPE
# =============================================================================

print("\n\n" + "=" * 70)
print("WEEKLY TREND BY MATERIAL TYPE")
print("=" * 70)

df_weekly_by_type = (
    df_weekly_stock
    .groupBy("Week_End", "Material_Type")
    .agg(F.sum("Stock_Value").alias("Total_Value_GBP"))
    .orderBy("Week_End", "Material_Type")
)

pdf_type_trend = df_weekly_by_type.toPandas()

if not pdf_type_trend.empty:
    # Pivot for stacked area chart
    pdf_pivot = pdf_type_trend.pivot(index='Week_End', columns='Material_Type', values='Total_Value_GBP').fillna(0)
    
    fig1 = go.Figure()
    
    colors = {'FERT': '#2E86AB', 'HALB': '#A23B72', 'ROH': '#F18F01'}
    
    for mat_type in pdf_pivot.columns:
        fig1.add_trace(go.Scatter(
            x=pdf_pivot.index,
            y=pdf_pivot[mat_type],
            mode='lines',
            name=mat_type,
            stackgroup='one',
            line=dict(width=0.5),
            fillcolor=colors.get(mat_type, '#888888')
        ))
    
    fig1.update_layout(
        title='Weekly Supply Chain Value by Material Type (GBP)',
        xaxis_title='Week Ending',
        yaxis_title='Total Value (£ GBP)',
        template='plotly_white',
        hovermode='x unified',
        yaxis_tickformat=',.0f',
        yaxis_tickprefix='£',
        legend_title='Material Type'
    )
    
    fig1.show()
    
    print("\n--- Weekly Value by Material Type ---")
    display(df_weekly_by_type)

# =============================================================================
# PART 2B: WEEKLY TREND BY LOCATION (PLANT)
# =============================================================================

print("\n\n" + "=" * 70)
print("WEEKLY TREND BY LOCATION (PLANT)")
print("=" * 70)

df_weekly_by_plant = (
    df_weekly_stock
    .groupBy("Week_End", "WERKS")
    .agg(F.sum("Stock_Value").alias("Total_Value_GBP"))
    .orderBy("Week_End", "WERKS")
)

pdf_plant_trend = df_weekly_by_plant.toPandas()

if not pdf_plant_trend.empty:
    # Pivot for multi-line chart
    pdf_plant_pivot = pdf_plant_trend.pivot(index='Week_End', columns='WERKS', values='Total_Value_GBP').fillna(0)
    
    fig2 = go.Figure()
    
    plant_colors = {'1000': '#2E86AB', '2000': '#A23B72', '3000': '#F18F01', '4000': '#4B8F29'}
    plant_names = {'1000': 'Plant 1000 (Hub)', '2000': 'Plant 2000', '3000': 'Plant 3000', '4000': 'Plant 4000'}
    
    for plant in pdf_plant_pivot.columns:
        fig2.add_trace(go.Scatter(
            x=pdf_plant_pivot.index,
            y=pdf_plant_pivot[plant],
            mode='lines+markers',
            name=plant_names.get(plant, f'Plant {plant}'),
            line=dict(color=plant_colors.get(plant, '#888888'), width=2),
            marker=dict(size=4)
        ))
    
    fig2.update_layout(
        title='Weekly Supply Chain Value by Location (GBP)',
        xaxis_title='Week Ending',
        yaxis_title='Total Value (£ GBP)',
        template='plotly_white',
        hovermode='x unified',
        yaxis_tickformat=',.0f',
        yaxis_tickprefix='£',
        legend_title='Plant'
    )
    
    fig2.show()
    
    print("\n--- Weekly Value by Plant ---")
    display(df_weekly_by_plant)

# =============================================================================
# PART 2C: TOTAL WEEKLY TREND
# =============================================================================

print("\n\n" + "=" * 70)
print("TOTAL WEEKLY SUPPLY CHAIN VALUE")
print("=" * 70)

df_weekly_total = (
    df_weekly_stock
    .groupBy("Week_End")
    .agg(F.sum("Stock_Value").alias("Total_Value_GBP"))
    .orderBy("Week_End")
)

pdf_total = df_weekly_total.toPandas()

if not pdf_total.empty:
    fig3 = go.Figure()
    
    fig3.add_trace(go.Scatter(
        x=pdf_total['Week_End'],
        y=pdf_total['Total_Value_GBP'],
        mode='lines+markers',
        name='Total Value',
        line=dict(color='#2E86AB', width=3),
        marker=dict(size=6),
        fill='tozeroy',
        fillcolor='rgba(46, 134, 171, 0.2)'
    ))
    
    fig3.update_layout(
        title='Total Weekly Supply Chain Value (GBP)',
        xaxis_title='Week Ending',
        yaxis_title='Total Value (£ GBP)',
        template='plotly_white',
        hovermode='x unified',
        yaxis_tickformat=',.0f',
        yaxis_tickprefix='£'
    )
    
    fig3.show()

In [ ]:
# CELL: PRODUCT PROFITABILITY ANALYSIS (P&L)
# -------------------------------------------------------------------------
# Goal: High-level product profitability - Revenue, COGS, Gross Profit
# Note: This excludes overheads, admin, people costs - purely product margin
# Revenue is based on actual sales order line values (VBAP.NETWR)
# -------------------------------------------------------------------------

import pyspark.sql.functions as F
from pyspark.sql.window import Window
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Load Tables
df_vbap = spark.table(f"{CATALOG}.{SCHEMA}.vbap")   # Sales order items (has price!)
df_vbak = spark.table(f"{CATALOG}.{SCHEMA}.vbak")   # Sales order headers
df_lips = spark.table(f"{CATALOG}.{SCHEMA}.lips")   # Delivery items (actual shipped)
df_vbfa = spark.table(f"{CATALOG}.{SCHEMA}.vbfa")   # Document flow (links orders to deliveries)
df_mbew = spark.table(f"{CATALOG}.{SCHEMA}.mbew")   # Material costs
df_mara = spark.table(f"{CATALOG}.{SCHEMA}.mara")   # Material master
df_makt = spark.table(f"{CATALOG}.{SCHEMA}.makt")   # Descriptions

print("=" * 70)
print("PRODUCT PROFITABILITY ANALYSIS")
print("=" * 70)
print("\nRevenue: Based on actual sales order line values (VBAP.NETWR)")
print("COGS: Based on standard cost (MBEW.STPRS)")

# =============================================================================
# PART 1: Calculate Sales & Profitability using VBFA Document Flow
# =============================================================================

# Use VBFA to link sales orders to deliveries
# VBELN = Sales Order, POSNN = SO Item, VBELN_N = Delivery, POSNN_N = Delivery Item
df_doc_flow = (
    df_vbfa
    .filter(F.col("VBTYP_V") == "C")  # C = Sales Order (source)
    .filter(F.col("VBTYP_N") == "J")  # J = Delivery (target)
    .select(
        F.col("VBELN").alias("Sales_Order"),
        F.col("POSNN").alias("Sales_Item"),
        F.col("VBELN_N").alias("Delivery"),
        F.col("POSNN_N").alias("Delivery_Item"),
        F.col("RFMNG").alias("Qty_Delivered")  # Quantity from doc flow
    )
)

# Join to VBAP to get pricing info
df_sales_with_price = (
    df_doc_flow.alias("f")
    .join(df_vbap.alias("v"),
          (F.col("f.Sales_Order") == F.col("v.VBELN")) &
          (F.col("f.Sales_Item") == F.col("v.POSNR")),
          how="inner")
    .select(
        F.col("v.MATNR"),
        F.col("v.WERKS"),
        F.col("f.Qty_Delivered"),
        # Calculate unit price from NETWR / KWMENG (line value / quantity)
        F.when(
            (F.col("v.KWMENG").isNotNull()) & (F.col("v.KWMENG") > 0),
            F.col("v.NETWR") / F.col("v.KWMENG")
        ).otherwise(F.lit(0)).alias("Unit_Price_GBP")
    )
)

# Aggregate by Material/Plant and join costs
df_sales_detail = (
    df_sales_with_price
    .groupBy("MATNR", "WERKS")
    .agg(
        F.sum("Qty_Delivered").alias("Qty_Sold"),
        F.avg("Unit_Price_GBP").alias("Avg_Unit_Price_GBP"),
        F.sum(F.col("Qty_Delivered") * F.col("Unit_Price_GBP")).alias("Revenue_GBP")
    )
    .alias("s")
    .join(df_mbew.alias("c"),
          (F.col("s.MATNR") == F.col("c.MATNR")) &
          (F.col("s.WERKS") == F.col("c.BWKEY")),
          how="left")
    .join(df_mara.alias("m"), F.col("s.MATNR") == F.col("m.MATNR"), how="left")
    .join(df_makt.alias("t"),
          (F.col("s.MATNR") == F.col("t.MATNR")) & (F.col("t.SPRAS") == "EN"),
          how="left")
    .select(
        F.col("s.WERKS").alias("Plant"),
        F.col("s.MATNR").alias("Material"),
        F.col("t.MAKTX").alias("Description"),
        F.col("m.MTART").alias("Material_Type"),
        F.col("s.Qty_Sold"),
        F.col("s.Avg_Unit_Price_GBP").alias("Unit_Price_GBP"),
        F.coalesce(F.col("c.STPRS"), F.lit(0)).alias("Unit_Cost_GBP"),
        F.col("s.Revenue_GBP")
    )
    .withColumn("COGS_GBP", F.col("Qty_Sold") * F.col("Unit_Cost_GBP"))
    .withColumn("Gross_Profit_GBP", F.col("Revenue_GBP") - F.col("COGS_GBP"))
    .withColumn("Gross_Margin_Pct",
                F.when(F.col("Revenue_GBP") > 0,
                       F.col("Gross_Profit_GBP") / F.col("Revenue_GBP") * 100)
                 .otherwise(F.lit(0)))
)

# =============================================================================
# PART 2: Summary P&L by Material Type
# =============================================================================

print("\n\n" + "=" * 70)
print("P&L SUMMARY BY MATERIAL TYPE")
print("=" * 70)

df_pl_by_type = (
    df_sales_detail
    .groupBy("Material_Type")
    .agg(
        F.sum("Qty_Sold").alias("Units_Sold"),
        F.sum("Revenue_GBP").alias("Revenue_GBP"),
        F.sum("COGS_GBP").alias("COGS_GBP"),
        F.sum("Gross_Profit_GBP").alias("Gross_Profit_GBP")
    )
    .withColumn("Gross_Margin_Pct",
                F.when(F.col("Revenue_GBP") > 0,
                       F.col("Gross_Profit_GBP") / F.col("Revenue_GBP") * 100)
                 .otherwise(F.lit(0)))
    .orderBy(F.col("Revenue_GBP").desc())
)

display(df_pl_by_type)

# =============================================================================
# PART 3: Summary P&L by Plant
# =============================================================================

print("\n" + "=" * 70)
print("P&L SUMMARY BY PLANT")
print("=" * 70)

df_pl_by_plant = (
    df_sales_detail
    .groupBy("Plant")
    .agg(
        F.countDistinct("Material").alias("Products_Sold"),
        F.sum("Qty_Sold").alias("Units_Sold"),
        F.sum("Revenue_GBP").alias("Revenue_GBP"),
        F.sum("COGS_GBP").alias("COGS_GBP"),
        F.sum("Gross_Profit_GBP").alias("Gross_Profit_GBP")
    )
    .withColumn("Gross_Margin_Pct",
                F.when(F.col("Revenue_GBP") > 0,
                       F.col("Gross_Profit_GBP") / F.col("Revenue_GBP") * 100)
                 .otherwise(F.lit(0)))
    .orderBy("Plant")
)

display(df_pl_by_plant)

# =============================================================================
# PART 4: Top/Bottom Products by Profitability
# =============================================================================

print("\n" + "=" * 70)
print("TOP 20 PRODUCTS BY GROSS PROFIT")
print("=" * 70)

df_top_products = (
    df_sales_detail
    .groupBy("Material", "Description", "Material_Type")
    .agg(
        F.sum("Qty_Sold").alias("Units_Sold"),
        F.sum("Revenue_GBP").alias("Revenue_GBP"),
        F.sum("COGS_GBP").alias("COGS_GBP"),
        F.sum("Gross_Profit_GBP").alias("Gross_Profit_GBP")
    )
    .withColumn("Gross_Margin_Pct",
                F.when(F.col("Revenue_GBP") > 0,
                       F.col("Gross_Profit_GBP") / F.col("Revenue_GBP") * 100)
                 .otherwise(F.lit(0)))
    .orderBy(F.col("Gross_Profit_GBP").desc())
)

display(df_top_products.limit(20))

# =============================================================================
# PART 5: GRAND TOTAL P&L
# =============================================================================

totals = df_sales_detail.agg(
    F.sum("Revenue_GBP").alias("Total_Revenue"),
    F.sum("COGS_GBP").alias("Total_COGS"),
    F.sum("Gross_Profit_GBP").alias("Total_Gross_Profit")
).collect()[0]

total_revenue = totals["Total_Revenue"] or 0
total_cogs = totals["Total_COGS"] or 0
total_gross_profit = totals["Total_Gross_Profit"] or 0
total_margin = (total_gross_profit / total_revenue * 100) if total_revenue > 0 else 0

print("\n" + "=" * 70)
print("COMPANY P&L SUMMARY (Product Level)")
print("=" * 70)
print(f"""
    Revenue:              £{total_revenue:>15,.2f}
    Cost of Goods Sold:   £{total_cogs:>15,.2f}
                          {'─' * 20}
    GROSS PROFIT:         £{total_gross_profit:>15,.2f}

    Gross Margin:         {total_margin:>15.1f}%
""")
print("=" * 70)
print("Note: Excludes overheads, admin, people costs, depreciation, etc.")
print("=" * 70)

# =============================================================================
# PART 6: Visualization - P&L Waterfall Chart
# =============================================================================

fig = go.Figure(go.Waterfall(
    name="P&L",
    orientation="v",
    measure=["absolute", "relative", "total"],
    x=["Revenue", "COGS", "Gross Profit"],
    y=[total_revenue, -total_cogs, total_gross_profit],
    text=[f"£{total_revenue:,.0f}", f"£{-total_cogs:,.0f}", f"£{total_gross_profit:,.0f}"],
    textposition="outside",
    connector={"line": {"color": "rgb(63, 63, 63)"}},
    decreasing={"marker": {"color": "#E74C3C"}},
    increasing={"marker": {"color": "#27AE60"}},
    totals={"marker": {"color": "#2E86AB"}}
))

fig.update_layout(
    title="Product P&L Waterfall (GBP)",
    template="plotly_white",
    showlegend=False,
    yaxis_tickformat=',.0f',
    yaxis_tickprefix='£'
)

fig.show()

# Margin by Plant Chart
pdf_plant = df_pl_by_plant.toPandas()

if not pdf_plant.empty:
    fig2 = make_subplots(specs=[[{"secondary_y": True}]])

    fig2.add_trace(
        go.Bar(x=pdf_plant['Plant'], y=pdf_plant['Revenue_GBP'], name='Revenue', marker_color='#2E86AB'),
        secondary_y=False
    )
    fig2.add_trace(
        go.Bar(x=pdf_plant['Plant'], y=pdf_plant['Gross_Profit_GBP'], name='Gross Profit', marker_color='#27AE60'),
        secondary_y=False
    )
    fig2.add_trace(
        go.Scatter(x=pdf_plant['Plant'], y=pdf_plant['Gross_Margin_Pct'], name='Margin %',
                   mode='lines+markers', line=dict(color='#E74C3C', width=3), marker=dict(size=10)),
        secondary_y=True
    )

    fig2.update_layout(
        title='Revenue & Profitability by Plant',
        template='plotly_white',
        barmode='group',
        legend=dict(orientation='h', yanchor='bottom', y=1.02)
    )
    fig2.update_yaxes(title_text="Value (£ GBP)", tickformat=',.0f', tickprefix='£', secondary_y=False)
    fig2.update_yaxes(title_text="Gross Margin %", ticksuffix='%', secondary_y=True)

    fig2.show()